In [49]:
import sys
sys.path.append("/Users/sujaladhikari/Desktop/FedIDS")

In [ ]:
import os 
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch 
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from FederatedLearningModel.fedmodel import MLP
from torch.optim import Adam
from sklearn.metrics import classification_report

In [51]:
dataset = pd.read_csv('../silos_datasets/combined_binary_silos.csv')
dataset = dataset.drop(columns = 'Unnamed: 0')
dataset.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,60453,94,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,0
1,53904,63,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,0
2,80,83165164,7,6,354,11595,348,0,50.571429,131.172732,...,20,51.0,0.0,51,51,83100000.0,0.0,83100000,83100000,1
3,80,85013328,6,6,361,11595,361,0,60.166667,147.377633,...,32,1014.0,0.0,1014,1014,84900000.0,0.0,84900000,84900000,1
4,80,83303368,8,5,337,11595,337,0,42.125000,119.147493,...,32,1011.0,0.0,1011,1011,83200000.0,0.0,83200000,83200000,1


In [52]:
print(dataset['Label_Binary'].value_counts())
print(f"The dataset has following number {dataset.shape[0]} number of rows and {dataset.shape[1]} number of columns")

Label_Binary
0    556488
1    556488
Name: count, dtype: int64
The dataset has following number 1112976 number of rows and 79 number of columns


In [53]:
random_seed = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [54]:
X = dataset.drop(columns = 'Label_Binary')
X = X.to_numpy()
y = dataset['Label_Binary']
y = y.to_numpy()


In [55]:
## Spliting the data into train, test, and validation splits
X_train , X_vals, y_train, y_vals = train_test_split(X,y , test_size=0.3, random_state=random_seed)
X_val, X_test, y_val, y_test = train_test_split(X_vals, y_vals, test_size=0.5, random_state=random_seed)

In [56]:
### Standard Scaling for all the datas

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_train = torch.tensor(X_train, dtype = torch.float32)
y_train = torch.tensor(y_train, dtype = torch.long)

X_val = scaler.transform(X_val)
X_val = torch.tensor(X_val, dtype = torch.float32)
y_val = torch.tensor(y_val, dtype = torch.long)

X_test = scaler.transform(X_test)
X_test = torch.tensor(X_test, dtype = torch.float32)
y_test = torch.tensor(y_test, dtype = torch.long)



In [57]:
training_data = TensorDataset(X_train,y_train)
testing_data = TensorDataset(X_test, y_test)
validation_data = TensorDataset(X_val, y_val)


In [58]:
training_batch = DataLoader(training_data, batch_size = 64, shuffle = True)
testing_batch = DataLoader(testing_data, batch_size=64, shuffle=False)
validation_batch = DataLoader(validation_data,batch_size = 64, shuffle = False)

In [59]:
print(MLP)

<class 'FederatedLearningModel.fedmodel.MLP'>


In [60]:
input_size = 78
hidden_layer = [256, 128,64,8]
num_classes = 2

In [61]:
## We will be using the same model that is being used in fedmodel.py for the federated purpose
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(input_size, hidden_layer, num_classes).to(device)
### Optimizer and loss function 
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
loss = nn.CrossEntropyLoss()
num_epoch = 10
model = model.to(device)

In [62]:
### Training, testing and evaluation of the centralized model 
## setting up the training environment:
def train(model, train_loader,validation_loader, criterion, device):
    model.train() ## Starting to train the model 
    training_loss = 0.0 ## Calculating the training loss in terms of floating value
    train_correct = 0 ## Count of total number of correct samples in total 
    training_total = 0 ## Count of the total number of samples checked
    for batch in train_loader: ## Loading the data loader
        samples, classes = batch 
        samples = samples.to(device) ## Converting the sample and classes to the device ('cpu')
        classes = classes.to(device)
        prediciton = model(samples) ## Prediction for the samples in the batch 
        optimizer.zero_grad() ## Started the calculation of the loss 
        loss = criterion(prediciton, classes)
        loss.backward() ## Backward propagation 
        optimizer.step() ## Forward optimizatio 
        training_loss +=loss.item() * samples.size(0) ## Calculating the loss for each batch 
        _,predicted = prediciton.max(1) ## Max prediction for each sample in the batch 
        training_total += classes.size(0) ### Each total in training adds up 
        train_correct += predicted.eq(classes).sum().item() ## Converting the total predicted that are equal to the classes and sum them 
    training_loss = training_loss/len(train_loader.dataset)
    training_accuracy = 100 * train_correct/ training_total 
    val_loss = 0
    val_correct = 0
    val_total = 0 
    model.eval()
    with torch.no_grad(): ## Evaluation started
        for val_batch in validation_loader:
            features, labels = val_batch 
            features = features.to(device) ## Converted the path to device 'cpu'
            labels = labels.to(device)
            val_outputs = model(features) ## Outputs
            val_loss_batch = criterion(val_outputs, labels) ## loss calculated
            val_loss += val_loss_batch.item() * features.size(0) ## for each batch 
            _, val_predicted = val_outputs.max(1) ## Maximum value calcuated
            val_correct += val_predicted.eq(labels).sum().item()
            val_total += labels.size(0) 
    avg_validation_loss = val_loss / len(validation_loader.dataset) ## Average across the sum 
    validation_accuracy = 100 * val_correct/ val_total ## Accuracy 
    return training_loss, training_accuracy , avg_validation_loss, validation_accuracy, val_correct, train_correct

In [63]:
### Creating the evalution function 
def evaluate(model, test_loader, critertion, device):
    model.eval()
    test_loss = 0.0 ## This is for the test loss 
    correct = 0 
    total = 0 
    predictions = []
    true_labels = []
    with torch.no_grad():
        for batch in test_loader:
            samples, labels = batch 
            samples = samples.to(device)
            labels = labels.to(device)
            outputs = model(samples)
            loss = critertion(outputs, labels)
            _, predicted = outputs.max(1)
            test_loss += loss.item() * samples.size(0)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            predictions.extend(predicted.tolist())
            true_labels.extend(labels.tolist())
        
    test_loss = test_loss / len(test_loader.dataset)
    accuracy = 100 * correct / total 

    return test_loss, accuracy, predictions, true_labels

### Since the training process has been done, we will evaluate the model's perfomance 

In [64]:
### On the given epoch we will be training and testing the epcoh 
num_epoch
for epoch in range(num_epoch):
    training_loss, training_accuracy , avg_validation_loss, validation_accuracy, val_correct, train_correct = train(model, training_batch,validation_batch, loss, device)
    test_loss, test_accuracy, predictions, true_labels = evaluate(model, testing_batch, loss, device)
    print(f"Epoch [{epoch+1}/{num_epoch}], Training Loss: {training_loss:4f}, Testing Loss: {test_loss:4f}")
    report = classification_report(true_labels, predictions)
    print(report)
    

Epoch [1/10], Training Loss: 0.108963, Testing Loss: 0.063330
              precision    recall  f1-score   support

           0       0.99      0.96      0.97     83612
           1       0.96      0.99      0.98     83335

    accuracy                           0.98    166947
   macro avg       0.98      0.98      0.98    166947
weighted avg       0.98      0.98      0.98    166947

Epoch [2/10], Training Loss: 0.900630, Testing Loss: 0.115745
              precision    recall  f1-score   support

           0       0.98      0.96      0.97     83612
           1       0.96      0.98      0.97     83335

    accuracy                           0.97    166947
   macro avg       0.97      0.97      0.97    166947
weighted avg       0.97      0.97      0.97    166947

Epoch [3/10], Training Loss: 0.074326, Testing Loss: 0.062790
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     83612
           1       0.97      0.99      0.98     833

### Lets evaluate how does the model do in each silos

In [108]:
def silos_evaluate(model, batch, criterion, device):
    model.eval()
    test_loss = 0.0
    total = 0 
    corrected = 0
    predictions = []
    true_labels = []
    for samples, features in batch:
        samples = samples.to(device)
        features = features.to(device)
        outputs = model(samples)
        loss = criterion(outputs, features)
        _, predicted = outputs.max(1)
        total += features.size(0)
        test_loss += loss.item() * samples.size(0)
        corrected += predicted.eq(features).sum().item()
        predictions.extend(predicted.tolist())
        true_labels.extend(features.tolist())
    
    test_loss = test_loss / len(batch.dataset)
    accuracy = 100 * corrected/total

    return test_loss, accuracy, predictions, true_labels


In [117]:
def batch_maker(dataset):
    dataset = dataset.drop(columns = 'Unnamed: 0', errors='ignore')
    X = dataset.drop(columns = 'Label_Binary')
    X = X.to_numpy()
    y = dataset['Label_Binary']
    y = y.to_numpy()
    X_train , X_test, y_train, y_test = train_test_split(X,y , test_size=0.3, random_state=random_seed)
    X_train = scaler.fit_transform(X_train)
    X_train = torch.tensor(X_train, dtype = torch.float32)
    y_train = torch.tensor(y_train, dtype = torch.long)

    X_test = scaler.transform(X_test)
    X_test = torch.tensor(X_test, dtype = torch.float32)
    y_test = torch.tensor(y_test, dtype = torch.long)

    training_batch = DataLoader(TensorDataset(X_train,y_train), batch_size = 64, shuffle = True)
    testing_batch = DataLoader(TensorDataset(X_test,y_test), batch_size=64, shuffle=False)
    
    return training_batch,testing_batch 

In [112]:
peter = pd.read_csv('../silos_datasets/SiloBinaryTwo.csv')
peter.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,52742,26932,2,1,44,6,38,6,22.0,22.627417,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
1,39839,52,1,1,0,0,0,0,0.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,0
2,5950,48,1,1,0,6,0,0,0.0,0.000000,...,40,0.0,0.0,0,0,0.0,0.0,0,0,1
3,53,411,2,2,94,542,47,47,47.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,0
4,705,11,1,1,2,6,2,2,2.0,0.000000,...,24,0.0,0.0,0,0,0.0,0.0,0,0,1


In [123]:
## Loading all the test data in the silos 
siloOne_train, siloOne_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryOne.csv'))
siloTwo_train, siloTwo_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryTwo.csv'))
siloThree_train, siloThree_test= batch_maker(pd.read_csv('../silos_datasets/siloBinaryThree.csv'))
siloFour_train, siloFour_test = batch_maker(pd.read_csv('../silos_datasets/siloBinaryFour.csv'))

In [122]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloOne_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(predictions, true_labels)
print(report)

 Test Loss: 0.7000, Test Accuracy: 83.0780%
              precision    recall  f1-score   support

           0       0.97      0.76      0.85     95890
           1       0.70      0.95      0.80     55138

    accuracy                           0.83    151028
   macro avg       0.83      0.86      0.83    151028
weighted avg       0.87      0.83      0.83    151028



In [124]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloTwo_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(predictions, true_labels)
print(report)

 Test Loss: 0.6511, Test Accuracy: 95.8874%
              precision    recall  f1-score   support

           0       0.93      0.99      0.96     81072
           1       0.99      0.93      0.96     92199

    accuracy                           0.96    173271
   macro avg       0.96      0.96      0.96    173271
weighted avg       0.96      0.96      0.96    173271



In [126]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloThree_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(predictions, true_labels)
print(report)

 Test Loss: 214.1072, Test Accuracy: 56.0723%
              precision    recall  f1-score   support

           0       0.83      0.54      0.66      6405
           1       0.29      0.62      0.39      1895

    accuracy                           0.56      8300
   macro avg       0.56      0.58      0.52      8300
weighted avg       0.70      0.56      0.60      8300



In [127]:
test_loss, test_accuracy, predictions, true_labels = silos_evaluate(model, siloFour_test, loss, device)
print(f" Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}%")
report = classification_report(predictions, true_labels)
print(report)

 Test Loss: 11.7102, Test Accuracy: 47.9167%
              precision    recall  f1-score   support

           0       0.83      0.50      0.62      1110
           1       0.11      0.38      0.17       186

    accuracy                           0.48      1296
   macro avg       0.47      0.44      0.40      1296
weighted avg       0.72      0.48      0.56      1296

